# 01 — Download & Merge CSV into one parquet file

<br>

- Dataset: **CSE-CIC-IDS2018**
- Task: multi-class classification of network attacks



<br>

<div align="center">

| Step | What |
|---|---|
| 1 | List the files in the public AWS bucket |
| 2 | Downloading |
| 3 | Load them in chunks, keeping all attacks and a sample of Benign |
| 4 | Explore: structure, classes, attacks per day |
| 5 | Quality checks: NaN, Inf, duplicates, constant columns |
| 6 | Save `combined_raw.parquet` to Drive for notebooks 01b and 02 |

</div>

- Detailed visualisations (distributions, boxplots, correlations, ports, PCA) are in **notebook 01b**.


In [1]:
!pip install -q boto3

In [4]:
# imports :
import os, sys
from pathlib import Path
from google.colab import drive
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import boto3
from botocore import UNSIGNED
from botocore.config import Config

COLLAB = True

In [5]:
# Mount drive :
if COLLAB :
  try:
      drive.mount("/content/drive")
      BASE = Path("/content/drive/MyDrive/Projects/CSE-CIC-IDS2018")
      print (BASE)
  except Exception as e :                      # running outside Colab
      print(f"Error {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Projects/CSE-CIC-IDS2018


In [6]:
# Define directories :
pd.set_option("display.max_columns", 100)

RAW_DIR = BASE / "data"
RAW_DIR.mkdir(parents=True, exist_ok=True)

## 1. Connect to S3 bucket & retrieve files :

The CSE-CIC-IDS2018 dataset contains two main categories of data:

- **Original Network Traffic and Log Data**: Includes PCAP archives containing raw network packets and log files. These can be used for deeper traffic analysis and custom feature extraction.

- **Processed Traffic Data for ML Algorithms**: Contains 10 CSV files (approximately 6.89 GB) with network-flow features extracted using CICFlowMeter.

In [6]:
BUCKET = "cse-cic-ids2018"
PREFIX = "Processed Traffic Data for ML Algorithms/"

# Connect to the public S3 bucket
try :
  s3 = boto3.client(
      "s3",
      region_name="ca-central-1",
      config=Config(signature_version=UNSIGNED)
  )
  print("Connected to S3 bucket")
except Exception as e :
  print(f"Error {e}")

Connected to S3 bucket


### Total files existing in the bucket :

In [8]:
response = s3.list_objects_v2(Bucket=BUCKET)

print("Total objects:", response.get("KeyCount", 0))

for obj in response.get("Contents", []):
    print(obj["Key"])

Total objects: 42
Original Network Traffic and Log data/
Original Network Traffic and Log data/Friday-02-03-2018/
Original Network Traffic and Log data/Friday-02-03-2018/logs.zip
Original Network Traffic and Log data/Friday-02-03-2018/pcap.zip
Original Network Traffic and Log data/Friday-16-02-2018/
Original Network Traffic and Log data/Friday-16-02-2018/logs.zip
Original Network Traffic and Log data/Friday-16-02-2018/pcap.zip
Original Network Traffic and Log data/Friday-23-02-2018/
Original Network Traffic and Log data/Friday-23-02-2018/logs.zip
Original Network Traffic and Log data/Friday-23-02-2018/pcap.zip
Original Network Traffic and Log data/Thursday-01-03-2018/
Original Network Traffic and Log data/Thursday-01-03-2018/logs.zip
Original Network Traffic and Log data/Thursday-01-03-2018/pcap.zip
Original Network Traffic and Log data/Thursday-15-02-2018/
Original Network Traffic and Log data/Thursday-15-02-2018/logs.zip
Original Network Traffic and Log data/Thursday-15-02-2018/pcap.

### Data Selection

For this project, we will use **the 10 processed CSV files as our primary data source**. They already contain extracted network-flow features and attack labels, making them suitable for supervised machine learning.

In [7]:
# Retrieve CSV files metadata
paginator = s3.get_paginator("list_objects_v2")

objects = []

for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".csv"):
            objects.append({
                "key": obj["Key"],
                "name": obj["Key"].removeprefix(PREFIX),
                "size_bytes": obj["Size"]
            })

# Create DataFrame
files = pd.DataFrame(objects)

if not files.empty:
    files["size_MB"] = (files["size_bytes"] / 1e6).round(1)

    total_size = files["size_bytes"].sum() / 1e9

    print(f"{len(files)} CSV files — {total_size:.2f} GB")
    display(files[["name", "size_MB"]])
else:
    print("No CSV files found.")

10 CSV files — 6.89 GB


,name,size_MB
0,Friday-02-03-2018_TrafficForML_CICFlowMeter.csv,352.4
1,Friday-16-02-2018_TrafficForML_CICFlowMeter.csv,333.7
2,Friday-23-02-2018_TrafficForML_CICFlowMeter.csv,382.8
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv,4054.9
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv,107.8
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv,375.9
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv,382.6
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter...,358.2
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter...,328.9
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter...,209.2


## 2. Download Files
`SELECTED = None` downloads all 10 days (needed to get **all attack types**, since each day has different attacks).
Use a list of indexes, e.g. `[0, 1, 2]`, for a quick test.

In [10]:
# Select files to download
selected_files = files.copy()  # All CSV files

# Download files
for row in selected_files.itertuples(index=False):

    file_path = RAW_DIR / row.name

    # Skip files that are already downloaded
    if file_path.exists() and file_path.stat().st_size == row.size_bytes:
        print(f"Already downloaded: {row.name}")
        continue

    print(f"Downloading: {row.name} ({row.size_MB} MB)")

    s3.download_file(BUCKET, row.key, str(file_path))

print("\nDownload completed!")

Downloading: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv (352.4 MB)
Downloading: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv (333.7 MB)
Downloading: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv (382.8 MB)
Downloading: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv (4054.9 MB)
Downloading: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv (107.8 MB)
Downloading: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv (375.9 MB)
Downloading: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv (382.6 MB)
Downloading: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv (358.2 MB)
Downloading: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv (328.9 MB)
Downloading: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv (209.2 MB)

Download completed!


### Inspect the CSV files shapes :

In [15]:
for file_path in sorted(RAW_DIR.glob("*.csv")):

    cols = len(pd.read_csv(file_path, nrows=0).columns)
    rows = sum(1 for _ in open(file_path, encoding="utf-8")) - 1

    print(f"\nFile: {file_path.name}")
    print(f"Rows: {rows:,}")
    print(f"Columns: {cols}")


File: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Rows: 1,048,575
Columns: 80

File: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Rows: 1,048,575
Columns: 80

File: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
Rows: 1,048,575
Columns: 80

File: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
Rows: 7,948,748
Columns: 84

File: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
Rows: 331,125
Columns: 80

File: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
Rows: 1,048,575
Columns: 80

File: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
Rows: 1,048,575
Columns: 80

File: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
Rows: 1,048,575
Columns: 80

File: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
Rows: 1,048,575
Columns: 80

File: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
Rows: 613,104
Columns: 80


## 3. Inspect one CSV file sample :

We gonna inspect a sample of 1000 rows :

In [19]:
sample_file = sorted(RAW_DIR.glob("*.csv"))[0]

df_sample = pd.read_csv(sample_file, nrows=1000, low_memory=False)

print("File:", sample_file.name)
print("Shape:", df_sample.shape)
df_sample.head(3)

File: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Shape: (1000, 80)


,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443,6,02/03/2018 08:47:38,141385,9,7,553,3773,202,0,61.444444,87.534438,1460,0,539.0,655.432936,30597.30523,113.166178,9425.666667,19069.116850,73403.0,1.0,141385.0,17673.125,23965.32327,73403.0,22.0,51417.0,8569.50000,13036.89082,31525.0,1.0,0,0,0,0,192,152,63.655975,49.510203,0,1460,254.470588,474.712955,225352.389700,0,0,1,1,0,0,0,1,0,270.375000,61.444444,539.0,0,0,0,0,0,0,9,553,7,3773,8192,119,4,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
1,49684,6,02/03/2018 08:47:38,281,2,1,38,0,38,0,19.000000,26.870058,0,0,0.0,0.000000,135231.31670,10676.156580,140.500000,174.655375,264.0,17.0,281.0,281.000,0.00000,281.0,281.0,0.0,0.00000,0.00000,0.0,0.0,1,0,0,0,40,20,7117.437722,3558.718861,0,38,19.000000,21.939310,481.333333,0,1,0,0,1,0,0,0,0,25.333333,19.000000,0.0,0,0,0,0,0,0,2,38,1,0,123,0,0,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign
2,443,6,02/03/2018 08:47:40,279824,11,15,1086,10527,385,0,98.727273,129.392497,1460,0,701.8,636.314186,41501.08640,92.915547,11192.960000,24379.448340,112589.0,1.0,279728.0,27972.800,36167.74032,112589.0,94.0,258924.0,18494.57143,36356.50372,133669.0,1.0,0,0,0,0,232,312,39.310424,53.605123,0,1460,430.111111,566.234209,320621.179500,0,0,1,1,0,0,0,1,1,446.653846,98.727273,701.8,0,0,0,0,0,0,11,1086,15,10527,8192,1047,5,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Benign


In [33]:
# Data type summary
print(df_sample.dtypes.value_counts().to_string())

int64      42
float64    36
object      2


In [38]:
for dt, cols in df_sample.columns.groupby(df_sample.dtypes.astype(str).values).items():
    print(f"\n{dt} ({len(cols)} feature)")
    display(pd.DataFrame(columns=cols))


float64 (36 feature)


feature,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Mean,Pkt Len Std,Pkt Len Var,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min



int64 (42 feature)


feature,Dst Port,Protocol,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Bwd Pkt Len Max,Bwd Pkt Len Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Pkt Len Min,Pkt Len Max,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min



object (2 feature)


feature,Timestamp,Label


## 4. Merging all CSV files :

### Check if all files got same structure

One file `Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv` is different from the others

In [46]:
csv_files = sorted(RAW_DIR.glob("*.csv"))

reference_columns = pd.read_csv(csv_files[0], nrows=0).columns

for file in csv_files[1:]:
    columns = pd.read_csv(file, nrows=0).columns

    if not columns.equals(reference_columns):
        print(f"Different structure: {file.name}")
    else:
        print(f"OK: {file.name}")

OK: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
OK: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
Different structure: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
OK: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
OK: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
OK: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
OK: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
OK: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
OK: Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv



The Tuesday (20-02-2018) CSV file contains 4 additional columns compared to the other files:

- `Flow ID`: Unique identifier for a network flow.
- `Src IP`: Source IP address.
- `Src Port`: Source port number.
- `Dst IP`: Destination IP address.

**These columns are not present in the other 9 CSV files.**

<br>

- We will remove these 4 columns during merging to ensure a consistent dataset structure.

- They are also not relevant for training, since they are identifiers or network-specific information rather than general traffic behavior features, which could lead to model overfitting


In [47]:
reference_columns = pd.read_csv(csv_files[0], nrows=0).columns
different_file = next(f for f in csv_files if "Thuesday-20" in f.name)

other_columns = pd.read_csv(different_file, nrows=0).columns

print("Missing columns:")
print(reference_columns.difference(other_columns).tolist())

print("\nExtra columns:")
print(other_columns.difference(reference_columns).tolist())

print("\nSame columns but different order?")
print(set(reference_columns) == set(other_columns))

Missing columns:
[]

Extra columns:
['Dst IP', 'Flow ID', 'Src IP', 'Src Port']

Same columns but different order?
False


### Merging as parquet file :

We will merge the 10 csv files as one parquet file

In [49]:
# Prepare directory
import pyarrow as pa
import pyarrow.parquet as pq

OUTPUT_DIR = BASE / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "CSE_CIC_IDS2018_merged.parquet"

In [50]:
# Merge using chunks :
csv_files = sorted(RAW_DIR.glob("*.csv"))

# Use the first file as the reference structure
reference_columns = pd.read_csv(csv_files[0], nrows=0).columns.tolist()

# Define a consistent Parquet schema
schema = pa.schema([
    pa.field(col, pa.string()) for col in reference_columns
])

total_rows = 0
chunk_size = 200_000

with pq.ParquetWriter(
    output_file,
    schema=schema,
    compression="snappy"
) as writer:

    for file in csv_files:
        print(f"\nProcessing: {file.name}")

        for chunk in pd.read_csv(
            file,
            chunksize=chunk_size,
            dtype="string",
            keep_default_na=False,
            usecols=lambda col: col in reference_columns
        ):

            # Ensure identical columns and order
            chunk = chunk[reference_columns]

            # Convert to Parquet and append
            table = pa.Table.from_pandas(
                chunk,
                schema=schema,
                preserve_index=False
            )

            writer.write_table(table)
            total_rows += len(chunk)

        print(f"Completed: {file.name}")

print("\nMerging completed!")
print(f"Total rows: {total_rows:,}")
print(f"Output: {output_file}")
print(f"File size: {output_file.stat().st_size / (1024**3):.2f} GB")


Processing: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
Completed: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv

Processing: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Completed: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv

Processing: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
Completed: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv

Processing: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
Completed: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv

Processing: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
Completed: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv

Processing: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
Completed: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv

Processing: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
Completed: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv

Processing: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
Completed: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv

Processing: Wedne

## 5. Inspect the merged parquet

In [9]:
import pandas as pd
import pyarrow.parquet as pq


output_file =  RAW_DIR / "processed" / "CSE_CIC_IDS2018_merged.parquet"

# Let's read the metadata of the parquet file to understand the schema and total row count without loading the whole file into memory
parquet_file = pq.ParquetFile(output_file)
print("Parquet Metadata:")
print(f"Total Row Groups: {parquet_file.num_row_groups}")
print(f"Total Rows: {parquet_file.metadata.num_rows:,}")
print(f"Number of Columns: {parquet_file.metadata.num_columns}")

Parquet Metadata:
Total Row Groups: 88
Total Rows: 16,233,002
Number of Columns: 80


In [15]:
# Read only a small sample (the first 10,000 rows) into a pandas DataFrame
df_preview = parquet_file.read_row_group(0).to_pandas().head(10000)

print("Preview Shape:", df_preview.shape)

Preview Shape: (10000, 80)


In [18]:
display(df_preview.head(3))

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,Fwd Pkt Len Mean,Fwd Pkt Len Std,Bwd Pkt Len Max,Bwd Pkt Len Min,Bwd Pkt Len Mean,Bwd Pkt Len Std,Flow Byts/s,Flow Pkts/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Tot,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Tot,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Pkt Len Min,Pkt Len Max,Pkt Len Mean,Pkt Len Std,Pkt Len Var,FIN Flag Cnt,SYN Flag Cnt,RST Flag Cnt,PSH Flag Cnt,ACK Flag Cnt,URG Flag Cnt,CWE Flag Count,ECE Flag Cnt,Down/Up Ratio,Pkt Size Avg,Fwd Seg Size Avg,Bwd Seg Size Avg,Fwd Byts/b Avg,Fwd Pkts/b Avg,Fwd Blk Rate Avg,Bwd Byts/b Avg,Bwd Pkts/b Avg,Bwd Blk Rate Avg,Subflow Fwd Pkts,Subflow Fwd Byts,Subflow Bwd Pkts,Subflow Bwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,443,6,02/03/2018 08:47:38,141385,9,7,553,3773,202,0,61.44444444,87.53443767,1460,0,539,655.4329358,30597.30523,113.1661775,9425.666667,19069.11685,73403,1,141385,17673.125,23965.32327,73403,22,51417,8569.5,13036.89082,31525,1,0,0,0,0,192,152,63.65597482,49.51020264,0,1460,254.4705882,474.7129551,225352.3897,0,0,1,1,0,0,0,1,0,270.375,61.44444444,539,0,0,0,0,0,0,9,553,7,3773,8192,119,4,20,0,0,0,0,0,0,0,0,Benign
1,49684,6,02/03/2018 08:47:38,281,2,1,38,0,38,0,19,26.87005769,0,0,0,0,135231.3167,10676.15658,140.5,174.655375,264,17,281,281,0,281,281,0,0,0,0,0,1,0,0,0,40,20,7117.437722,3558.718861,0,38,19,21.93931023,481.3333333,0,1,0,0,1,0,0,0,0,25.33333333,19,0,0,0,0,0,0,0,2,38,1,0,123,0,0,20,0,0,0,0,0,0,0,0,Benign
2,443,6,02/03/2018 08:47:40,279824,11,15,1086,10527,385,0,98.72727273,129.3924966,1460,0,701.8,636.3141856,41501.0864,92.91554692,11192.96,24379.44834,112589,1,279728,27972.8,36167.74032,112589,94,258924,18494.57143,36356.50372,133669,1,0,0,0,0,232,312,39.31042369,53.60512322,0,1460,430.1111111,566.234209,320621.1795,0,0,1,1,0,0,0,1,1,446.6538462,98.72727273,701.8,0,0,0,0,0,0,11,1086,15,10527,8192,1047,5,20,0,0,0,0,0,0,0,0,Benign
